# Team02 FastAPI + ngrok Relay for the Final SageMaker Champion Endpoint

This notebook creates a temporary **FastAPI relay inside the Team02 SageMaker Studio/Jupyter environment** and exposes that relay through **ngrok** for the ITI113 classroom demo.

It is synchronized to:

- **Source MLOps notebook:** `S201_1552444F_03_sagemaker_pipeline_team02_champion_model_UPDATED_FINAL_v5.ipynb`
- **Pipeline / Model Package Group:** `team02-diabetes-risk`
- **Final SageMaker endpoint:** `team02-diabetes-risk`
- **Selected champion:** XGBoost
- **Champion final MLflow run:** `057bb58d81564c0db9a628b224eab8f9`
- **Decision threshold:** `0.50`
- **Champion config version:** `2026-08-22-cross-model-champion-v5`
- **Data version:** `brfss2015-diabetes-binary-6244bec277fe`

## Recommended demo architecture

```text
Local / external Streamlit app
        ↓ HTTPS + x-api-key
ngrok public URL
        ↓
FastAPI relay running inside Team02 SageMaker Studio
        ↓ boto3 using the SageMaker execution role
AWS SageMaker Serverless Endpoint
team02-diabetes-risk
        ↓
XGBoost champion response + model/data/config lineage
```

### Why this design is preferred

The Streamlit computer does **not** need AWS Access Key / Secret Access Key credentials.

The FastAPI relay runs inside SageMaker Studio/Jupyter, where `boto3` uses the attached SageMaker execution role to invoke the final champion endpoint.

The Streamlit app needs only:

1. the temporary ngrok `/predict` URL; and
2. the temporary **Relay API Key** entered securely in this notebook.

> **Important:** The Relay API Key is **not obtained from AWS**. It is a temporary shared secret chosen for the demo.

> Use this relay only for temporary project testing/demo. Stop ngrok and FastAPI after the demonstration.


## Security and lineage controls for final submission

This notebook deliberately separates **external demo credentials** from **AWS credentials**.

Security controls:
- reads the temporary Relay API Key from `RELAY_API_KEY` or prompts with `getpass`;
- never prints the Relay API Key;
- never stores AWS Access Key / Secret Access Key in the notebook;
- fixes the SageMaker endpoint server-side as `team02-diabetes-risk`;
- validates the exact 21-feature request contract;
- validates returned **model type, model run ID, config version, data version and decision threshold** against Notebook 03 v5;
- writes only sanitized evidence artifacts;
- keeps AWS credentials inside the SageMaker environment;
- requires the user to stop ngrok and FastAPI after the demo.

These controls strengthen **C — MLOps & Deployment** through deployment traceability and **E — AI Governance** through controlled access, data minimisation, human-operated exposure and model-version checks.


## Champion handoff synchronized from Notebook 03 v5

The relay must not silently point to a stale Progress Check endpoint or a different model version.

The frozen values below come from the final-champion configuration in Notebook 03 v5:

| Item | Final value |
|---|---|
| Endpoint | `team02-diabetes-risk` |
| Model family | `XGBoost` |
| Final model run ID | `057bb58d81564c0db9a628b224eab8f9` |
| Decision threshold | `0.50` |
| Config version | `2026-08-22-cross-model-champion-v5` |
| Data version | `brfss2015-diabetes-binary-6244bec277fe` |
| Held-out PR-AUC | `0.464397478` |
| ROC-AUC | `0.824820173` |
| F1 | `0.492441889` |
| Recall | `0.621388368` |
| Precision | `0.407814809` |

The relay uses these values as **deployment lineage assertions**. A prediction is rejected with HTTP 502 if the endpoint responds with stale or mismatched lineage metadata.

This does not replace formal model evaluation; it verifies that the demo application is calling the intended approved champion.


## 1. Confirm that this notebook is running with an AWS identity

Run this notebook **inside the Team02 SageMaker Studio/Jupyter environment**.

If this cell succeeds, the relay can use the attached SageMaker execution role.  
If it fails with `NoCredentialsError`, you are probably running the notebook outside AWS and should move the relay back into SageMaker Studio.

In [ ]:
import boto3
import json
from botocore.exceptions import NoCredentialsError, ClientError

try:
    sts = boto3.client("sts")
    identity = sts.get_caller_identity()
    print("AWS identity available:")
    print(json.dumps(identity, indent=2))
except NoCredentialsError:
    raise RuntimeError(
        "No AWS credentials are available. Run this relay notebook inside "
        "the Team02 SageMaker Studio/Jupyter environment."
    )

## 2. Team02 final-champion configuration

Notebook 03 v5 defines the production-style classroom demo endpoint as:

- **Region:** `ap-southeast-1`
- **Endpoint:** `team02-diabetes-risk`
- **Model Package Group:** `team02-diabetes-risk`
- **Type:** SageMaker Serverless Inference
- **Selected model:** XGBoost
- **Decision threshold:** `0.50`
- **Expected endpoint status:** `InService`

The older Progress Check endpoint `team02-diabetes-risk-test` is **not** used by this final relay.

The Relay API Key is a separate temporary shared secret between Streamlit and FastAPI. It is not an AWS credential.


In [ ]:
import os
from getpass import getpass

REGION = "ap-southeast-1"
ENDPOINT_NAME = "team02-diabetes-risk"
MODEL_PACKAGE_GROUP = "team02-diabetes-risk"

# Frozen champion lineage from Notebook 03 v5.
EXPECTED_MODEL_TYPE = "XGBoost"
EXPECTED_MODEL_RUN_ID = "057bb58d81564c0db9a628b224eab8f9"
EXPECTED_DECISION_THRESHOLD = 0.50
EXPECTED_CONFIG_VERSION = "2026-08-22-cross-model-champion-v5"
EXPECTED_DATA_VERSION = "brfss2015-diabetes-binary-6244bec277fe"

# Shared secret between Streamlit and the FastAPI relay.
# NEVER print or save the value in notebook output.
RELAY_API_KEY = os.getenv("RELAY_API_KEY", "").strip()

if not RELAY_API_KEY:
    RELAY_API_KEY = getpass(
        "Enter a temporary relay shared secret of at least 24 characters (input hidden): "
    ).strip()

if len(RELAY_API_KEY) < 24:
    raise ValueError(
        "Use a temporary relay secret of at least 24 characters."
    )

print("Region             :", REGION)
print("Endpoint           :", ENDPOINT_NAME)
print("Model package group:", MODEL_PACKAGE_GROUP)
print("Expected model     :", EXPECTED_MODEL_TYPE)
print("Expected run ID    :", EXPECTED_MODEL_RUN_ID)
print("Expected threshold :", EXPECTED_DECISION_THRESHOLD)
print("Expected config    :", EXPECTED_CONFIG_VERSION)
print("Expected data      :", EXPECTED_DATA_VERSION)
print("Relay API key configured securely: YES (value not displayed)")


## 3. Confirm the final SageMaker endpoint is available

This preflight verifies:

1. the Team02 SageMaker execution role can describe the endpoint;
2. the endpoint is exactly `team02-diabetes-risk`;
3. the endpoint is `InService`;
4. its endpoint configuration is readable; and
5. serverless deployment details are captured as sanitized evidence.

This should pass **before** FastAPI or ngrok is started.


In [ ]:
import boto3
import json
from datetime import datetime, timezone
from pathlib import Path
from botocore.exceptions import ClientError

sm = boto3.client("sagemaker", region_name=REGION)

try:
    endpoint = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    endpoint_status = endpoint["EndpointStatus"]

    print("Endpoint status :", endpoint_status)
    print("Endpoint ARN    :", endpoint["EndpointArn"])
    print("Endpoint config :", endpoint["EndpointConfigName"])
    print("Creation time   :", endpoint.get("CreationTime"))
    print("Last modified   :", endpoint.get("LastModifiedTime"))

    endpoint_config = sm.describe_endpoint_config(
        EndpointConfigName=endpoint["EndpointConfigName"]
    )

    variants = endpoint_config.get("ProductionVariants", [])
    if not variants:
        raise RuntimeError("Endpoint configuration has no ProductionVariants.")

    print("\nProduction variants:")
    for variant in variants:
        print(
            json.dumps(
                {
                    "VariantName": variant.get("VariantName"),
                    "ModelName": variant.get("ModelName"),
                    "ServerlessConfig": variant.get("ServerlessConfig"),
                },
                indent=2,
                default=str,
            )
        )

    if endpoint_status != "InService":
        raise RuntimeError(
            f"Final endpoint is not InService. Current status: {endpoint_status}"
        )

    endpoint_preflight_evidence = {
        "captured_at_utc": datetime.now(timezone.utc).isoformat(),
        "region": REGION,
        "endpoint_name": ENDPOINT_NAME,
        "endpoint_arn": endpoint["EndpointArn"],
        "endpoint_status": endpoint_status,
        "endpoint_config_name": endpoint["EndpointConfigName"],
        "production_variants": [
            {
                "variant_name": v.get("VariantName"),
                "model_name": v.get("ModelName"),
                "serverless_config": v.get("ServerlessConfig"),
            }
            for v in variants
        ],
        "expected_champion": {
            "model_type": EXPECTED_MODEL_TYPE,
            "model_run_id": EXPECTED_MODEL_RUN_ID,
            "decision_threshold": EXPECTED_DECISION_THRESHOLD,
            "config_version": EXPECTED_CONFIG_VERSION,
            "data_version": EXPECTED_DATA_VERSION,
        },
    }

    Path("team02_endpoint_preflight_evidence.json").write_text(
        json.dumps(endpoint_preflight_evidence, indent=2, default=str),
        encoding="utf-8",
    )
    print("\nSaved sanitized evidence: team02_endpoint_preflight_evidence.json")

except ClientError as e:
    print("Could not describe final endpoint.")
    print("Code   :", e.response["Error"]["Code"])
    print("Message:", e.response["Error"]["Message"])
    raise


## 4. Define the Team02 diabetes request and champion response contracts

The final endpoint expects the same **21 BRFSS-derived features** used by Notebook 03 v5.

The relay also checks the response metadata produced by the v5 `inference.py` handler. This provides an operational guard against accidentally calling a stale endpoint/model.

The sample below is only a technical smoke-test record and is **not** a clinical example.


In [ ]:
import math
import json

FEATURE_COLUMNS = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]

EXPECTED_RESPONSE_FIELDS = {
    "screening_class",
    "screening_category",
    "probability",
    "decision_threshold",
    "model_type",
    "data_version",
    "model_run_id",
    "config_version",
    "intended_use_notice",
}

sample_payload = {
    "HighBP": 0,
    "HighChol": 0,
    "CholCheck": 1,
    "BMI": 25.0,
    "Smoker": 0,
    "Stroke": 0,
    "HeartDiseaseorAttack": 0,
    "PhysActivity": 1,
    "Fruits": 1,
    "Veggies": 1,
    "HvyAlcoholConsump": 0,
    "AnyHealthcare": 1,
    "NoDocbcCost": 0,
    "GenHlth": 2,
    "MentHlth": 0,
    "PhysHlth": 0,
    "DiffWalk": 0,
    "Sex": 1,
    "Age": 8,
    "Education": 5,
    "Income": 6,
}

assert list(sample_payload.keys()) == FEATURE_COLUMNS

def validate_champion_response(payload):
    if not isinstance(payload, list) or len(payload) != 1:
        raise RuntimeError(
            "Final endpoint should return a one-row JSON list for a single request."
        )

    row = payload[0]
    missing = EXPECTED_RESPONSE_FIELDS - set(row)
    if missing:
        raise RuntimeError(
            f"Endpoint response is missing expected fields: {sorted(missing)}"
        )

    if row["model_type"] != EXPECTED_MODEL_TYPE:
        raise RuntimeError(
            f"Model mismatch: expected {EXPECTED_MODEL_TYPE}, got {row['model_type']}"
        )

    if row["model_run_id"] != EXPECTED_MODEL_RUN_ID:
        raise RuntimeError(
            "Model run/version mismatch: "
            f"expected {EXPECTED_MODEL_RUN_ID}, got {row['model_run_id']}"
        )

    if row["config_version"] != EXPECTED_CONFIG_VERSION:
        raise RuntimeError(
            "Config version mismatch: "
            f"expected {EXPECTED_CONFIG_VERSION}, got {row['config_version']}"
        )

    if row["data_version"] != EXPECTED_DATA_VERSION:
        raise RuntimeError(
            "Data version mismatch: "
            f"expected {EXPECTED_DATA_VERSION}, got {row['data_version']}"
        )

    if not math.isclose(
        float(row["decision_threshold"]),
        EXPECTED_DECISION_THRESHOLD,
        rel_tol=0.0,
        abs_tol=1e-12,
    ):
        raise RuntimeError(
            "Decision-threshold mismatch: "
            f"expected {EXPECTED_DECISION_THRESHOLD}, "
            f"got {row['decision_threshold']}"
        )

    probability = float(row["probability"])
    if not 0.0 <= probability <= 1.0:
        raise RuntimeError(f"Invalid probability returned: {probability}")

    expected_class = int(probability >= EXPECTED_DECISION_THRESHOLD)
    if int(row["screening_class"]) != expected_class:
        raise RuntimeError(
            "Endpoint screening_class is inconsistent with probability and threshold."
        )

    return row

print("Request contract contains", len(sample_payload), "features.")
print(json.dumps(sample_payload, indent=2))
print("\nExpected champion response fields:")
print(sorted(EXPECTED_RESPONSE_FIELDS))


## 5. Test direct final-endpoint invocation from SageMaker Studio

This is the key diagnostic before introducing FastAPI/ngrok:

```text
SageMaker notebook role
        ↓
SageMaker Runtime
        ↓
team02-diabetes-risk
        ↓
XGBoost champion metadata assertion
```

A successful HTTP invocation alone is not sufficient. The response must also match the frozen Notebook 03 v5 model/data/config lineage.


In [ ]:
import boto3
import json
from pathlib import Path
from datetime import datetime, timezone
from botocore.exceptions import ClientError, NoCredentialsError

runtime = boto3.client("sagemaker-runtime", region_name=REGION)

try:
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(sample_payload).encode("utf-8"),
    )

    raw_result = response["Body"].read().decode("utf-8")
    print("Raw endpoint response:")
    print(raw_result)

    parsed_result = json.loads(raw_result)
    champion_row = validate_champion_response(parsed_result)

    print("\nFINAL CHAMPION RESPONSE VALIDATION: PASS")
    print("Model type        :", champion_row["model_type"])
    print("Model run ID      :", champion_row["model_run_id"])
    print("Config version    :", champion_row["config_version"])
    print("Data version      :", champion_row["data_version"])
    print("Decision threshold:", champion_row["decision_threshold"])
    print("Probability       :", champion_row["probability"])
    print("Screening class   :", champion_row["screening_class"])

    direct_evidence = {
        "captured_at_utc": datetime.now(timezone.utc).isoformat(),
        "endpoint_name": ENDPOINT_NAME,
        "region": REGION,
        "request_feature_count": len(sample_payload),
        "response": champion_row,
        "lineage_validation": "PASS",
    }
    Path("team02_direct_endpoint_evidence.json").write_text(
        json.dumps(direct_evidence, indent=2, default=str),
        encoding="utf-8",
    )
    print("\nSaved sanitized evidence: team02_direct_endpoint_evidence.json")

except NoCredentialsError:
    raise RuntimeError(
        "AWS credentials are not available. Run this notebook inside "
        "the Team02 SageMaker environment."
    )
except ClientError as e:
    print("Endpoint invocation failed.")
    print("Code   :", e.response["Error"]["Code"])
    print("Message:", e.response["Error"]["Message"])
    raise


## 6. Install relay dependencies

Run this once in the SageMaker notebook environment.

In [ ]:
%pip install -q fastapi "uvicorn[standard]" pyngrok requests "pydantic>=2"

## 7. Create the final-champion FastAPI relay

The relay exposes:

- `GET /` — non-secret service metadata;
- `GET /health` — authenticated endpoint health check;
- `POST /predict` — authenticated prediction relay with champion-lineage validation;
- `GET /docs` — FastAPI interactive documentation.

### Security / governance controls

- the external caller cannot choose an arbitrary SageMaker endpoint;
- `/health` and `/predict` require `x-api-key`;
- the payload is validated against the exact 21 expected features;
- additional fields are rejected;
- non-finite values are rejected;
- the SageMaker response must match the frozen v5 model/data/config lineage;
- the relay returns model output but never returns AWS credentials;
- CORS middleware is not enabled because the Streamlit app calls the relay server-side.


In [ ]:
%%writefile relay_api.py
"""Authenticated Team02 relay for the final SageMaker champion endpoint."""

import hmac
import json
import math
import os
from typing import Any

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError, NoCredentialsError
from fastapi import Depends, FastAPI, HTTPException
from fastapi.security import APIKeyHeader
from pydantic import BaseModel, ConfigDict, Field

REGION = os.environ.get("AWS_REGION", "ap-southeast-1")
ENDPOINT_NAME = os.environ.get("ENDPOINT_NAME", "team02-diabetes-risk")
RELAY_API_KEY = os.environ.get("RELAY_API_KEY", "")

EXPECTED_MODEL_TYPE = os.environ.get("EXPECTED_MODEL_TYPE", "XGBoost")
EXPECTED_MODEL_RUN_ID = os.environ.get(
    "EXPECTED_MODEL_RUN_ID",
    "057bb58d81564c0db9a628b224eab8f9",
)
EXPECTED_DECISION_THRESHOLD = float(
    os.environ.get("EXPECTED_DECISION_THRESHOLD", "0.50")
)
EXPECTED_CONFIG_VERSION = os.environ.get(
    "EXPECTED_CONFIG_VERSION",
    "2026-08-22-cross-model-champion-v5",
)
EXPECTED_DATA_VERSION = os.environ.get(
    "EXPECTED_DATA_VERSION",
    "brfss2015-diabetes-binary-6244bec277fe",
)

EXPECTED_RESPONSE_FIELDS = {
    "screening_class",
    "screening_category",
    "probability",
    "decision_threshold",
    "model_type",
    "data_version",
    "model_run_id",
    "config_version",
    "intended_use_notice",
}

if not RELAY_API_KEY:
    raise RuntimeError(
        "RELAY_API_KEY is missing. Set it before starting the FastAPI relay."
    )

app = FastAPI(
    title="Team02 Diabetes Final-Champion SageMaker Relay",
    version="2.0",
    description=(
        "Authenticated FastAPI relay for the Team02 ITI113 diabetes "
        "screening-support final SageMaker champion endpoint."
    ),
)

api_key_header = APIKeyHeader(name="x-api-key", auto_error=False)

runtime = boto3.client(
    "sagemaker-runtime",
    region_name=REGION,
    config=Config(
        connect_timeout=10,
        read_timeout=70,
        retries={"max_attempts": 2, "mode": "standard"},
    ),
)

sagemaker = boto3.client(
    "sagemaker",
    region_name=REGION,
    config=Config(
        connect_timeout=10,
        read_timeout=20,
        retries={"max_attempts": 2, "mode": "standard"},
    ),
)


class DiabetesFeatures(BaseModel):
    model_config = ConfigDict(
        extra="forbid",
        allow_inf_nan=False,
    )

    HighBP: int = Field(ge=0, le=1)
    HighChol: int = Field(ge=0, le=1)
    CholCheck: int = Field(ge=0, le=1)
    BMI: float = Field(gt=0)
    Smoker: int = Field(ge=0, le=1)
    Stroke: int = Field(ge=0, le=1)
    HeartDiseaseorAttack: int = Field(ge=0, le=1)
    PhysActivity: int = Field(ge=0, le=1)
    Fruits: int = Field(ge=0, le=1)
    Veggies: int = Field(ge=0, le=1)
    HvyAlcoholConsump: int = Field(ge=0, le=1)
    AnyHealthcare: int = Field(ge=0, le=1)
    NoDocbcCost: int = Field(ge=0, le=1)
    GenHlth: int = Field(ge=1, le=5)
    MentHlth: int = Field(ge=0, le=30)
    PhysHlth: int = Field(ge=0, le=30)
    DiffWalk: int = Field(ge=0, le=1)
    Sex: int = Field(ge=0, le=1)
    Age: int = Field(ge=1, le=13)
    Education: int = Field(ge=1, le=6)
    Income: int = Field(ge=1, le=8)


def require_api_key(
    supplied_key: str | None = Depends(api_key_header),
) -> None:
    if supplied_key is None or not hmac.compare_digest(
        supplied_key,
        RELAY_API_KEY,
    ):
        raise HTTPException(status_code=401, detail="Invalid relay API key.")


def validate_champion_response(payload: Any) -> list[dict[str, Any]]:
    """Fail closed if SageMaker serves stale or unexpected model lineage."""
    if not isinstance(payload, list) or not payload:
        raise HTTPException(
            status_code=502,
            detail="SageMaker returned an unexpected response shape.",
        )

    for row in payload:
        if not isinstance(row, dict):
            raise HTTPException(
                status_code=502,
                detail="SageMaker returned a non-object prediction row.",
            )

        missing = EXPECTED_RESPONSE_FIELDS - set(row)
        if missing:
            raise HTTPException(
                status_code=502,
                detail=(
                    "SageMaker response is missing expected fields: "
                    + ", ".join(sorted(missing))
                ),
            )

        lineage_mismatches = {}

        if row.get("model_type") != EXPECTED_MODEL_TYPE:
            lineage_mismatches["model_type"] = {
                "expected": EXPECTED_MODEL_TYPE,
                "actual": row.get("model_type"),
            }

        if row.get("model_run_id") != EXPECTED_MODEL_RUN_ID:
            lineage_mismatches["model_run_id"] = {
                "expected": EXPECTED_MODEL_RUN_ID,
                "actual": row.get("model_run_id"),
            }

        if row.get("config_version") != EXPECTED_CONFIG_VERSION:
            lineage_mismatches["config_version"] = {
                "expected": EXPECTED_CONFIG_VERSION,
                "actual": row.get("config_version"),
            }

        if row.get("data_version") != EXPECTED_DATA_VERSION:
            lineage_mismatches["data_version"] = {
                "expected": EXPECTED_DATA_VERSION,
                "actual": row.get("data_version"),
            }

        try:
            actual_threshold = float(row.get("decision_threshold"))
        except (TypeError, ValueError):
            actual_threshold = math.nan

        if not math.isclose(
            actual_threshold,
            EXPECTED_DECISION_THRESHOLD,
            rel_tol=0.0,
            abs_tol=1e-12,
        ):
            lineage_mismatches["decision_threshold"] = {
                "expected": EXPECTED_DECISION_THRESHOLD,
                "actual": row.get("decision_threshold"),
            }

        try:
            probability = float(row.get("probability"))
        except (TypeError, ValueError):
            probability = math.nan

        if not math.isfinite(probability) or not 0.0 <= probability <= 1.0:
            lineage_mismatches["probability"] = {
                "expected": "finite value between 0 and 1",
                "actual": row.get("probability"),
            }
        else:
            expected_class = int(probability >= EXPECTED_DECISION_THRESHOLD)
            if int(row.get("screening_class", -1)) != expected_class:
                lineage_mismatches["screening_class"] = {
                    "expected": expected_class,
                    "actual": row.get("screening_class"),
                }

        if lineage_mismatches:
            raise HTTPException(
                status_code=502,
                detail={
                    "message": (
                        "Deployed endpoint metadata does not match the frozen "
                        "Notebook 03 v5 champion contract."
                    ),
                    "mismatches": lineage_mismatches,
                },
            )

    return payload


@app.get("/")
def home() -> dict[str, Any]:
    return {
        "status": "running",
        "service": "Team02 Diabetes Final-Champion SageMaker Relay",
        "relay_version": "2.0",
        "endpoint": ENDPOINT_NAME,
        "region": REGION,
        "expected_model_type": EXPECTED_MODEL_TYPE,
        "expected_model_run_id": EXPECTED_MODEL_RUN_ID,
        "expected_config_version": EXPECTED_CONFIG_VERSION,
        "routes": [
            "GET /health",
            "POST /predict",
            "GET /docs",
        ],
    }


@app.get("/health")
def health(_: None = Depends(require_api_key)) -> dict[str, Any]:
    try:
        desc = sagemaker.describe_endpoint(EndpointName=ENDPOINT_NAME)
        return {
            "relay": "ok",
            "relay_version": "2.0",
            "endpoint_name": ENDPOINT_NAME,
            "endpoint_status": desc.get("EndpointStatus"),
            "endpoint_config_name": desc.get("EndpointConfigName"),
            "region": REGION,
            "expected_champion": {
                "model_type": EXPECTED_MODEL_TYPE,
                "model_run_id": EXPECTED_MODEL_RUN_ID,
                "decision_threshold": EXPECTED_DECISION_THRESHOLD,
                "config_version": EXPECTED_CONFIG_VERSION,
                "data_version": EXPECTED_DATA_VERSION,
            },
            "prediction_lineage_check": (
                "enforced on POST /predict"
            ),
        }

    except NoCredentialsError:
        raise HTTPException(
            status_code=503,
            detail=(
                "AWS credentials are not available to the relay. "
                "Run this relay inside the Team02 SageMaker Studio/Jupyter environment."
            ),
        )

    except ClientError as exc:
        code = exc.response.get("Error", {}).get("Code", "ClientError")
        message = exc.response.get("Error", {}).get("Message", "")
        raise HTTPException(
            status_code=502,
            detail=f"SageMaker health check failed: {code}: {message}",
        )


@app.post("/predict")
def predict(
    features: DiabetesFeatures,
    _: None = Depends(require_api_key),
) -> Any:
    payload = features.model_dump()

    try:
        response = runtime.invoke_endpoint(
            EndpointName=ENDPOINT_NAME,
            ContentType="application/json",
            Accept="application/json",
            Body=json.dumps(payload).encode("utf-8"),
        )

        raw = response["Body"].read().decode("utf-8")

        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            raise HTTPException(
                status_code=502,
                detail="SageMaker returned a non-JSON response.",
            )

        return validate_champion_response(parsed)

    except NoCredentialsError:
        raise HTTPException(
            status_code=503,
            detail=(
                "AWS credentials are not available to the relay. "
                "Run this relay inside the Team02 SageMaker Studio/Jupyter environment."
            ),
        )

    except ClientError as exc:
        code = exc.response.get("Error", {}).get("Code", "ClientError")
        message = exc.response.get("Error", {}).get("Message", "")
        raise HTTPException(
            status_code=502,
            detail=f"SageMaker invocation failed: {code}: {message}",
        )


## 8. Start the FastAPI relay inside SageMaker

The server listens on port `8000`.

The notebook passes the fixed final endpoint name, expected champion lineage and temporary Relay API Key through environment variables.

The secret value is never printed.


In [ ]:
import os
import subprocess
import time

os.environ["AWS_REGION"] = REGION
os.environ["ENDPOINT_NAME"] = ENDPOINT_NAME
os.environ["RELAY_API_KEY"] = RELAY_API_KEY
os.environ["EXPECTED_MODEL_TYPE"] = EXPECTED_MODEL_TYPE
os.environ["EXPECTED_MODEL_RUN_ID"] = EXPECTED_MODEL_RUN_ID
os.environ["EXPECTED_DECISION_THRESHOLD"] = str(EXPECTED_DECISION_THRESHOLD)
os.environ["EXPECTED_CONFIG_VERSION"] = EXPECTED_CONFIG_VERSION
os.environ["EXPECTED_DATA_VERSION"] = EXPECTED_DATA_VERSION

# Stop a previous relay process if this cell is re-run.
try:
    relay_process.terminate()
    relay_process.wait(timeout=5)
    print("Stopped previous FastAPI server.")
except Exception:
    pass

relay_process = subprocess.Popen(
    [
        "uvicorn",
        "relay_api:app",
        "--host", "0.0.0.0",
        "--port", "8000",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(3)

if relay_process.poll() is not None:
    print("FastAPI failed to start. Output:")
    print(relay_process.stdout.read())
    raise RuntimeError("FastAPI relay did not start.")

print("FastAPI final-champion relay started.")
print("PID:", relay_process.pid)
print("Local root:    http://127.0.0.1:8000/")
print("Local health:  http://127.0.0.1:8000/health")
print("Local docs:    http://127.0.0.1:8000/docs")
print("Local predict: http://127.0.0.1:8000/predict")


## 9. Test the local relay before using ngrok

Test `/health` first.

A successful response should identify:

```json
{
  "relay": "ok",
  "relay_version": "2.0",
  "endpoint_name": "team02-diabetes-risk",
  "endpoint_status": "InService",
  "region": "ap-southeast-1",
  "expected_champion": {
    "model_type": "XGBoost",
    "model_run_id": "057bb58d81564c0db9a628b224eab8f9",
    "decision_threshold": 0.5,
    "config_version": "2026-08-22-cross-model-champion-v5",
    "data_version": "brfss2015-diabetes-binary-6244bec277fe"
  }
}
```

`/health` confirms endpoint availability. The stronger model-lineage assertion is enforced during `/predict`.


In [ ]:
import requests
import json

local_health_url = "http://127.0.0.1:8000/health"

headers = {
    "x-api-key": RELAY_API_KEY,
    "Content-Type": "application/json",
}

response = requests.get(
    local_health_url,
    headers=headers,
    timeout=30,
)

print("Status code:", response.status_code)
try:
    print(json.dumps(response.json(), indent=2))
except Exception:
    print(response.text)

response.raise_for_status()

### Test local `/predict`

This verifies the complete path:

```text
FastAPI → boto3 → SageMaker endpoint → FastAPI
```

In [ ]:
local_predict_url = "http://127.0.0.1:8000/predict"

response = requests.post(
    local_predict_url,
    headers=headers,
    json=sample_payload,
    timeout=75,
)

print("Status code:", response.status_code)
try:
    local_prediction = response.json()
    print(json.dumps(local_prediction, indent=2))
except Exception:
    print(response.text)
    raise

response.raise_for_status()

# The relay itself already validates lineage; validate again in the notebook
# so the execution evidence is explicit.
local_champion_row = validate_champion_response(local_prediction)
print("\nLocal FastAPI → SageMaker champion-lineage validation: PASS")
print("Model run ID:", local_champion_row["model_run_id"])
print("Config      :", local_champion_row["config_version"])


## 10. Configure ngrok

Use your ngrok **authtoken** here.

The ngrok authtoken and Relay API Key are two different credentials:

| Credential | Purpose |
|---|---|
| ngrok authtoken | Allows this notebook to create an ngrok tunnel |
| Relay API Key | Protects `/health` and `/predict` from unauthorised callers |

The Relay API Key is entered securely in Section 2 and is **not supplied by AWS**.

`getpass()` prevents the ngrok token from appearing directly in the notebook cell.

In [ ]:
from getpass import getpass
from pyngrok import ngrok

NGROK_AUTH_TOKEN = getpass("Paste ngrok auth token: ").strip()

if not NGROK_AUTH_TOKEN:
    raise ValueError("No ngrok auth token supplied.")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok auth token configured for this notebook session.")

## 11. Start the public ngrok tunnel

Only the temporary FastAPI relay on port `8000` is exposed.

The Streamlit application will call:

```text
https://<ngrok-domain>/predict
```

In [ ]:
from pyngrok import ngrok
import time

print("Stopping any ngrok process managed by this notebook...")

try:
    ngrok.kill()
except Exception as e:
    print("No local ngrok process to stop:", e)

time.sleep(3)

print("Starting fresh ngrok tunnel on FastAPI port 8000...")

public_tunnel = ngrok.connect(
    addr=8000,
    proto="http"
)

public_url = public_tunnel.public_url.rstrip("/")

RELAY_HEALTH_URL = public_url + "/health"
RELAY_PREDICT_URL = public_url + "/predict"

print("=" * 72)
print("NGROK TUNNEL STARTED")
print("=" * 72)
print("Public relay root:")
print(public_url)

print("\nHealth URL:")
print(RELAY_HEALTH_URL)

print("\nPrediction URL:")
print(RELAY_PREDICT_URL)

print("\nFastAPI docs:")
print(public_url + "/docs")
print("=" * 72)

In [ ]:
from pyngrok import ngrok
import requests
import time

print("=" * 72)
print("NGROK DIAGNOSTIC")
print("=" * 72)

# 1. Check FastAPI is still alive
try:
    r = requests.get(
        "http://127.0.0.1:8000/",
        timeout=10,
    )
    print("Local FastAPI root:", r.status_code)
except Exception as e:
    print("Local FastAPI FAILED:", repr(e))

# 2. Check ngrok agent process
try:
    proc = ngrok.get_ngrok_process()
    print("ngrok process object:", proc)
    print("ngrok process running:", proc.proc.poll() is None)
    print("ngrok PID:", proc.proc.pid)
except Exception as e:
    print("Could not inspect ngrok process:", repr(e))

# 3. Ask local ngrok agent which tunnels it believes are active
try:
    tunnels = ngrok.get_tunnels()
    print("\nActive tunnels:", len(tunnels))

    for tunnel in tunnels:
        print("Public URL :", tunnel.public_url)
        print("Target     :", tunnel.config.get("addr"))
except Exception as e:
    print("Could not retrieve tunnels:", repr(e))

# 4. Test current public URL
try:
    print("\nTesting:", public_url)
    r = requests.get(
        public_url + "/",
        headers={"ngrok-skip-browser-warning": "true"},
        timeout=20,
    )
    print("Public HTTP:", r.status_code)
    print(r.text[:500])
except Exception as e:
    print("Public test failed:", repr(e))

## 12. Test the public ngrok relay end to end

This simulates the same path used by the external Streamlit application:

```text
external caller
    ↓ HTTPS / x-api-key
ngrok
    ↓
FastAPI relay
    ↓ AWS execution role
team02-diabetes-risk
    ↓
v5 champion lineage validation
```

The cell records sanitized evidence only. No Relay API Key or ngrok token is written to disk.


In [ ]:
headers = {
    "x-api-key": RELAY_API_KEY,
    "Content-Type": "application/json",
    "ngrok-skip-browser-warning": "true",
}

print("Testing public health route...")
health_response = requests.get(
    RELAY_HEALTH_URL,
    headers=headers,
    timeout=30,
)

print("Health status:", health_response.status_code)
try:
    public_health = health_response.json()
    print(json.dumps(public_health, indent=2))
except Exception:
    print(health_response.text)
    raise

health_response.raise_for_status()

if public_health.get("endpoint_name") != ENDPOINT_NAME:
    raise RuntimeError(
        f"Public relay points to {public_health.get('endpoint_name')} "
        f"instead of {ENDPOINT_NAME}."
    )
if public_health.get("endpoint_status") != "InService":
    raise RuntimeError(
        f"Public relay endpoint is not InService: {public_health}"
    )

print("\nTesting public prediction route...")
predict_response = requests.post(
    RELAY_PREDICT_URL,
    headers=headers,
    json=sample_payload,
    timeout=75,
)

print("Prediction status:", predict_response.status_code)
try:
    public_result = predict_response.json()
    print(json.dumps(public_result, indent=2))
except Exception:
    print(predict_response.text)
    raise

predict_response.raise_for_status()

public_champion_row = validate_champion_response(public_result)
print("\nPUBLIC RELAY CHAMPION-LINEAGE VALIDATION: PASS")

# Save sanitized public-relay evidence. No secret is written.
from datetime import datetime, timezone
from urllib.parse import urlsplit
from pathlib import Path

sanitized_relay_host = urlsplit(RELAY_PREDICT_URL).netloc

relay_evidence = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "relay_host": sanitized_relay_host,
    "health_http_status": health_response.status_code,
    "prediction_http_status": predict_response.status_code,
    "sagemaker_endpoint": ENDPOINT_NAME,
    "region": REGION,
    "expected_model_type": EXPECTED_MODEL_TYPE,
    "expected_model_run_id": EXPECTED_MODEL_RUN_ID,
    "expected_decision_threshold": EXPECTED_DECISION_THRESHOLD,
    "expected_config_version": EXPECTED_CONFIG_VERSION,
    "expected_data_version": EXPECTED_DATA_VERSION,
    "prediction_response": public_result,
    "lineage_validation": "PASS",
    "credential_handling": (
        "x-api-key used at runtime; secret value intentionally not recorded"
    ),
}

Path("team02_public_relay_evidence.json").write_text(
    json.dumps(relay_evidence, indent=2, default=str),
    encoding="utf-8",
)
print("\nSaved sanitized evidence: team02_public_relay_evidence.json")


## 13. Values to enter in the Team02 Streamlit app

Your Streamlit application should use:

- **Relay predict URL:** the current ngrok URL ending in `/predict`
- **Relay API Key:** the same temporary secret entered securely in Section 2

It should **not** use AWS Access Keys and should **not** call SageMaker directly.

The relay is fixed to `team02-diabetes-risk`, so the external UI cannot redirect the backend to a different SageMaker endpoint.


In [ ]:
print("=" * 72)
print("STREAMLIT CONNECTION SETTINGS")
print("=" * 72)
print("Relay Predict URL:")
print(RELAY_PREDICT_URL)
print()
print("Relay API Key:")
print("<use the SAME temporary secret supplied securely in Section 2>")
print()
print("Security note: the secret is intentionally not printed by this notebook.")
print("=" * 72)


### Optional: configure the Streamlit app through PowerShell environment variables

If you are running `team02_streamlit_app.py` on your Windows computer, use:

```powershell
$env:RELAY_PREDICT_URL="<COPY THE NGROK /predict URL FROM THIS NOTEBOOK>"
$env:RELAY_API_KEY="<COPY THE RELAY API KEY FROM THIS NOTEBOOK>"

streamlit run team02_streamlit_app.py
```

Notice that there is **no local AWS credential configuration** in this architecture.

The AWS call is made by the relay inside SageMaker Studio.

In [ ]:
powershell_example = f'''$env:RELAY_PREDICT_URL="{RELAY_PREDICT_URL}"
$env:RELAY_API_KEY="<PASTE_THE_SAME_TEMPORARY_SECRET>"

streamlit run team02_streamlit_app.py
'''

print(powershell_example)


## 14. External caller example

The external UI needs only:

- Relay `/predict` URL
- Relay API Key
- Correct 21-feature JSON payload

It does not need the AWS account ID, IAM role, Access Key ID, Secret Access Key, or Session Token.

In [ ]:
example_code = f'''import requests

url = "{RELAY_PREDICT_URL}"

headers = {{
    "x-api-key": "<PASTE_THE_SAME_TEMPORARY_SECRET>",
    "Content-Type": "application/json",
}}

payload = {json.dumps(sample_payload, indent=4)}

response = requests.post(
    url,
    headers=headers,
    json=payload,
    timeout=75,
)

print(response.status_code)
print(response.json())
'''

print(example_code)


## 15. Final demo architecture and assessment evidence

For the ITI113 Final Report, this notebook demonstrates the application/deployment path:

```text
Streamlit client
  ↓ HTTPS / x-api-key
ngrok
  ↓
FastAPI authenticated relay
  ↓ boto3 / SageMaker execution role
SageMaker Serverless Endpoint
team02-diabetes-risk
  ↓
XGBoost final champion
  ↓
probability + thresholded screening result
  + model/data/config lineage
```

### Final champion synchronized from Notebook 03 v5

- model family: **XGBoost**;
- final model run ID: `057bb58d81564c0db9a628b224eab8f9`;
- threshold: **0.50**;
- config version: `2026-08-22-cross-model-champion-v5`;
- data version: `brfss2015-diabetes-binary-6244bec277fe`;
- held-out PR-AUC: **0.464397478**.

### Demonstrated backend evidence

- fixed final SageMaker endpoint integration;
- serverless endpoint preflight and configuration capture;
- direct endpoint invocation;
- exact v5 champion response-lineage assertions;
- authenticated FastAPI relay;
- strict 21-feature request validation;
- local and public relay prediction tests;
- separation of external client credentials from AWS credentials;
- sanitized endpoint/direct/public evidence JSON artifacts;
- documented shutdown procedure.

### Final UI evidence still required

Run the corrected Streamlit app, make one successful prediction, save its end-to-end evidence artifact, and capture one screenshot showing the result. This proves:

**Streamlit → FastAPI relay → SageMaker final champion → response → Streamlit UI**

Do not fabricate or manually edit that evidence.

### AI governance implications

- **Human oversight:** this relay is started/stopped deliberately for the demo; it does not auto-deploy or auto-retrain a model.
- **Traceability:** every prediction must match the expected model run, config version, data version and threshold.
- **Data minimisation:** only the 21 approved model features are accepted; extra fields are rejected.
- **Credential separation:** the client never receives AWS credentials.
- **Intended use:** the model response states that this is an academic screening-support prototype and **not a diagnosis**.
- **Monitoring linkage:** Notebook 03 v5 remains the source of pipeline, registry, approval, deployment and drift-monitoring evidence.

This supports **C — MLOps & Deployment** and **E — AI Governance**.


## 16. Stop ngrok and FastAPI after the demo

Run this immediately when testing/presentation is complete.

This is a deliberate **human operational control** that reduces unnecessary public exposure and avoids accidental endpoint calls after the classroom demo.


In [ ]:
# Stop ngrok tunnels created by pyngrok.
try:
    ngrok.kill()
    print("ngrok tunnels stopped.")
except Exception as e:
    print("ngrok stop issue:", e)

# Stop FastAPI relay.
try:
    relay_process.terminate()
    relay_process.wait(timeout=5)
    print("FastAPI relay stopped.")
except Exception as e:
    print("FastAPI stop issue:", e)

## 17. Requirements / rubric evidence from this relay notebook

### Requirement 1 — end-to-end flow linkage

Notebook 03 v5 covers:

**ingestion → preprocessing → training → evaluation → MLflow tracking → Model Registry → human approval → deployment → inference → monitoring**

This relay notebook extends the deployed-inference path:

**Streamlit → ngrok → FastAPI → SageMaker final endpoint → lineage-validated response**

### Requirement 2 — reproducible operational evidence

The relay records or validates:

- endpoint name and endpoint configuration;
- final model run/version;
- champion config version;
- data version;
- decision threshold;
- request feature schema;
- prediction response;
- sanitized evidence timestamps;
- no saved Relay API Key or AWS credentials.

### Requirement 3 — complete MLOps/deployment setup

The complete project evidence is split intentionally:

- **Notebook 03 v5:** data pipeline, experiment tracking, Model Registry, manual approval, deployment, CI/CD template, CloudWatch/PSI/performance/config drift monitoring;
- **this relay notebook:** application integration and authenticated external inference path;
- **Streamlit app:** user-facing screening-support workflow.

### Governance alignment

- human-over-the-loop deployment remains in Notebook 03 v5;
- relay fails closed on stale model/data/config lineage;
- only intended features are accepted;
- credentials are separated;
- public exposure is temporary and human-controlled;
- output retains the screening-support / not-diagnosis notice.


# Troubleshooting

## Streamlit says:
`{"detail":"AWS credentials are not available to the relay."}`

The **FastAPI relay** is running in an environment without AWS credentials.

### Fix

Run this relay notebook inside the Team02 SageMaker Studio/Jupyter environment.  
Your Windows/local Streamlit client should call the ngrok URL and should not host the SageMaker relay unless AWS credentials have deliberately been configured there.

---

## `401 Invalid relay API key`

The Relay API Key is the temporary shared secret entered in Section 2.

It does **not** come from AWS.

Use exactly the same secret in the Streamlit application.

---

## HTTP `502` with `Deployed endpoint metadata does not match...`

This is an intentional fail-closed MLOps control.

The relay expected the Notebook 03 v5 champion:

- endpoint: `team02-diabetes-risk`
- model: `XGBoost`
- model run: `057bb58d81564c0db9a628b224eab8f9`
- threshold: `0.50`
- config: `2026-08-22-cross-model-champion-v5`
- data version: `brfss2015-diabetes-binary-6244bec277fe`

If the response differs, verify that the correct approved Model Registry package is deployed to the final endpoint. Do not bypass the check merely to make the demo pass.

---

## `AccessDeniedException`

The SageMaker execution role has AWS credentials but lacks permission for the required SageMaker actions.

At minimum, the relay needs permission to:

- `sagemaker:DescribeEndpoint`
- `sagemaker:DescribeEndpointConfig`
- `sagemaker:InvokeEndpoint`

Scope the permission to the Team02 resources where possible.

---

## `ValidationException`, HTTP 422, missing fields or extra fields

Send exactly the 21 Team02 diabetes features.

Do not send old heart-disease prototype fields such as:

```text
cp
trestbps
chol
thalach
oldpeak
ca
thal
```

The final v5 endpoint and relay use the BRFSS-derived diabetes feature schema.

---

## Endpoint is not `InService`

Check the **final** endpoint:

```python
sm.describe_endpoint(
    EndpointName="team02-diabetes-risk"
)
```

Do not use the older Progress Check endpoint `team02-diabetes-risk-test` for the final demo.

---

## ngrok URL changes

A temporary/free ngrok URL can change whenever the tunnel is restarted.

Update `RELAY_PREDICT_URL` in Streamlit after restarting ngrok.

---

## ngrok token security

Do not hard-code your ngrok authtoken into the notebook.

Use `getpass()` as implemented here. Rotate any token that has been exposed in chat, Git, screenshots or submitted files.

---

## Final demo sequence

1. Open this notebook in Team02 SageMaker Studio/Jupyter.
2. Run the AWS identity and final-endpoint preflight sections.
3. Run the direct SageMaker invocation and confirm **champion-lineage validation PASS**.
4. Start FastAPI.
5. Test local `/health` and `/predict`.
6. Configure ngrok and start the tunnel.
7. Test public `/health` and `/predict`.
8. Confirm **PUBLIC RELAY CHAMPION-LINEAGE VALIDATION: PASS**.
9. Configure Streamlit with only the ngrok `/predict` URL and Relay API Key.
10. Submit one screening record and capture the result/evidence.
11. Stop ngrok and FastAPI after the demo.
